<a href="https://colab.research.google.com/github/Cairo-Henrique/Movie-Recommender/blob/main/movies_embeddings_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movies Embeddings Creation

This notebook contains a pipeline for generating vector embeddings from a movie dataset using Natural Language Processing models. The generated embeddings are used to build a content-based recommendation engine powered by Cosine Similarity.

## Overview

The primary goal of this notebook is to process movie metadata, including titles, overviews, and genres. It converts these text fields into dense vector representations and calculates similarity scores to recommend movies based on free-text input.

## Features and Methodology

The notebook offers two distinct strategies for creating and evaluating embeddings:

1.  **combined_texts (Holistic Approach)**
    * Concatenates the title, overview, and genres into a single string.
    * Generates a single embedding for the entire movie profile.
    * Compares user input against this single combined vector.

2.  **title_overview_genres (Granular Approach)**
    * Generates three separate embeddings for each movie: one for the title, one for the overview, and one for the genres.
    * Calculates independent cosine similarities for each of these three categories.
    * Merges the similarities using a weighted linear combination.
    * The default weights prioritize the plot and title: $0.35 \cdot \text{Title} + 0.50 \cdot \text{Overview} + 0.15 \cdot \text{Genre}$.

## Models Used

The code includes configurations to test different architectures:
* **Sentence Transformers:** Uses `all-mpnet-base-v2` for optimized sentence-level semantic representations.
* **BERT:** Includes a custom function `get_movie_embedding_BERT` to tokenize and pool embeddings using standard BERT models.

According to some tests, the best model is `all-mpnet-base-v2`.

## Dependencies

The following libraries are required to run the notebook:
* `pandas`
* `numpy`
* `scikit-learn`
* `sentence-transformers`
* `torch`

## Pipeline Steps

1.  **Environment Setup:** Installs PyTorch with CUDA support and manages conflicting packages.
2.  **Function Definitions:** Sets up vectorization functions and the core recommendation engine.
3.  **Data Loading:** Reads `movies_fixed.csv` and cleans missing values in the metadata columns.
4.  **Embedding Generation:** Processes text data through the chosen model in batches to append high-dimensional vectors to the DataFrame.
5.  **Interactive Testing:** Prompts for a text description, converts it to an embedding, and returns the top 5 most similar movies.
6.  **Export:** Saves the final dataset and computed embeddings as a serialized pickle file for production use.

* **Input:** `movies_fixed.csv`
* **Output:** `movies_with_embeddings_all-mpnet-base-v2.pkl`

## Packages

In [ ]:
!pip uninstall torch torchvision torchaudio -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124


In [ ]:
!pip uninstall torchcodec -y

Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128


In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

from sklearn.metrics.pairwise import cosine_similarity

#from transformers import BertTokenizer, BertModel
from sentence_transformers import SentenceTransformer, util

import torch

## Functions

In [ ]:
def get_all_embeddings(movies_dataset: pd.DataFrame, style: str, show_progress_bar=True):
    if style == 'combined_texts':
        combined_texts = (
        movies_dataset['title'] + " " +
        movies_dataset['overview'] + " " +
        movies_dataset['genres']).tolist()
        return list(model.encode(combined_texts, show_progress_bar=show_progress_bar))
    elif style == 'title_overview_genres':
        titles = model.encode(movies_dataset['title'].tolist(), show_progress_bar=show_progress_bar)
        overviews = model.encode(movies_dataset['overview'].tolist(), show_progress_bar=show_progress_bar)
        genres = model.encode(movies_dataset['genres'].tolist(), show_progress_bar=show_progress_bar)
        return list(titles), list(overviews), list(genres)

def get_overall_similarities(sim_title, sim_overview, sim_genres):
    #sim = (sim_title + sim_overview + sim_genres) / 3
    #sim = np.maximum(np.maximum(sim_title, sim_overview), sim_genres)
    sim = 0.35 * sim_title + 0.5 * sim_overview + 0.15 * sim_genres
    return sim

def recommend_movies(input_embedding, movies: pd.DataFrame, style: str, top_n=5):

    if style == 'combined_texts':

        # Similaridade com embedding combinado
        similarities = cosine_similarity([input_embedding], movies['embedding'])[0]

    elif style == 'title_overview_genres':

        # Similaridades separadas
        title_sim = cosine_similarity(
            [input_embedding],
            np.vstack(movies['embedding_title'])
        )[0]

        overview_sim = cosine_similarity(
            [input_embedding],
            np.vstack(movies['embedding_overview'])
        )[0]

        genres_sim = cosine_similarity(
            [input_embedding],
            np.vstack(movies['embedding_genres'])
        )[0]

        # Média das similaridades
        similarities = get_overall_similarities(title_sim, overview_sim, genres_sim)

    movies_copy = movies.copy()
    movies_copy['similarity'] = similarities

    # Excluir filmes sem overview
    movies_copy = movies_copy[movies_copy['overview'].str.strip() != ""]

    # Excluir filmes sem genres
    movies_copy = movies_copy[movies_copy['genres'].str.strip() != ""]

    recommended = movies_copy.sort_values(
        by='similarity', ascending=False
    ).head(top_n)

    return recommended[['title', 'overview', 'similarity']]

def get_movie_embedding_BERT(title, overview, genres):
    # Gera embedding usando BERT
    text = f"title: {title}. overview: {overview}. genres: {genres}"
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding

## Import `movies_fixed.csv` dataset

In [ ]:
# Caminho do arquivo
movies_path = '/content/movies_fixed.csv'

# Leitura do arquivo
movies = pd.read_csv(movies_path, encoding='utf-8')

# Garantir que todas as colunas existem e estão tratadas
movies['title'] = movies['title'].fillna('').astype(str)
movies['overview'] = movies['overview'].fillna('').astype(str)
movies['genres'] = movies['genres'].fillna('').astype(str)

## BERT

In [ ]:
# Carregar modelo BERT pré-treinado
model_name = 'bert-case-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

In [ ]:
# Gerar embeddings considerando title + overview + genres
movies['embedding'] = movies.apply(
    lambda row: get_movie_embedding_BERT(row['title'], row['overview'], row['genres']),
    axis=1
)

## Sentence Transformers

In [ ]:
# Carregar o modelo SentenceTransformer

model_name = 'all-mpnet-base-v2'
#model_name = 'all-MiniLM-L6-v2'

model = SentenceTransformer(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Escolher estilo
#style = 'combined_texts' # creates embeddings for one string containing title, overview, and genres
style = 'title_overview_genres' # creates 1 embedding for each one: title, overview, and genres

In [ ]:
# Gerar embeddings de forma eficiente (em batch)

if style == 'combined_texts':
    movies['embedding'] = get_all_embeddings(movies, style)
elif style == 'title_overview_genres':
    movies['embedding_title'], movies['embedding_overview'], movies['embedding_genres'] = get_all_embeddings(movies, style)

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

## Testar e salvar

In [ ]:
# Input do usuário
input_description = input("Enter a movie description: ")

# Generate input embedding using the SentenceTransformer model
input_embedding = model.encode(input_description)

# Generate input embedding using the BERT model
#input_embedding = get_movie_embedding_BERT(input_description, input_description, input_description)

# Obter recomendações
recommended_movies = recommend_movies(input_embedding, movies, style, top_n=5)

print("Recommended Movies:")
print(recommended_movies)

Enter a movie description: star galaxy epic
Recommended Movies:
                                                  title  \
7888                                       galaxy quest   
4082  the legend of the galactic heroes: die neue th...   
2117                                   star trek beyond   
9005                         stargate: the ark of truth   
7400                               the last starfighter   

                                               overview  similarity  
7888  for four years, the courageous crew of the nse...    0.467472  
4082  in humanity's distant future, two interstellar...    0.457979  
2117  the uss enterprise crew explores the furthest ...    0.442785  
9005  sg-1 searches for an ancient weapon which coul...    0.442516  
7400  a video game expert alex rogan finds himself t...    0.442514  


In [ ]:
# Salva embeddings
movies.to_pickle(f"/content/movies_with_embeddings_{model_name}.pkl")